# **LangChain — Parte 2**

Este notebook reúne e adapta vários tutoriais em um único material de estudo. O conteúdo foi reorganizado para facilitar a leitura progressiva, com explicações didáticas curtas e separação por temas.

Ao longo do material, você verá exemplos sobre:

1. **Fundamentos de memória e contexto**: `runtime context` e `state`.
2. **Orquestração**: uso de subagentes e coordenação.
3. **Ciclo de execução do agente**: mensagens, aprovações humanas e middleware.
4. **Comportamento dinâmico**: prompts, tools e escolha de modelos.
5. **Aplicações práticas**: RAG, SQL e um caso de uso mais completo.

In [9]:
from dotenv import load_dotenv

load_dotenv()

True

In [10]:
from langchain.chat_models import init_chat_model
import os

model = init_chat_model(
    model="gpt-5-nano",
    model_provider="azure_openai",
    azure_deployment=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
)

# **Runtime Context**

## **O que é `context_schema`**

Neste exemplo, o agente recebe informações de contexto que não precisam necessariamente ficar persistidas como histórico conversacional. Isso é útil quando você quer injetar dados de execução, preferências do usuário ou parâmetros operacionais.

In [34]:
from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "azul"
    least_favourite_colour: str = "amarelo"


## **Criando um agente que recebe context**

Aqui o agente é criado com `context_schema=ColourContext`. Isso permite passar um objeto de contexto no momento da chamada.

In [35]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    context_schema=ColourContext
)

In [36]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Qual é a minha cor favorita?")]},
    context=ColourContext()
)

In [37]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Qual é a minha cor favorita?', additional_kwargs={}, response_metadata={}, id='20d60ab8-827a-4e26-88d0-32a08cb35c73'),
              AIMessage(content='Não sei qual é a sua cor favorita, pois isso é pessoal. Se quiser, posso tentar adivinhar com algumas pistas.\n\nQue tal um mini quiz rápido? Responda às perguntas com uma opção:\n\n1) Você prefere cores frias (azul, verde, roxo) ou quentes (vermelho, laranja, amarelo)?\n2) Você gosta de cores claras ou escuras?\n3) Você gosta de cores fortes e marcantes ou de cores suaves e pastéis?\n\nCom base nas suas respostas eu digo qual pode ser a sua cor favorita. Se preferir, me diga apenas uma pista direto (ex.: “minha cor favorita é azul”).', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1491, 'prompt_tokens': 13, 'total_tokens': 1504, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1344, 'rejected_predictio

## **Acessando o contexto dentro de tools**

O passo seguinte é expor esse `context` para tools por meio de `ToolRuntime`. Repare que a leitura passa a acontecer diretamente em `runtime.context`.

In [19]:
from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime) -> str:
    """Obtém a cor favorita do usuário"""
    return runtime.context.favourite_colour

@tool
def get_least_favourite_colour(runtime: ToolRuntime) -> str:
    """Obtém a cor menos favorita do usuário"""
    return runtime.context.least_favourite_colour

In [15]:
agent = create_agent(
    model=model,
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext
)

In [16]:
response = agent.invoke(
    {"messages": [HumanMessage(content="Qual é a minha cor favorita?")]},
    context=ColourContext()
)

pprint(response)

/home/willemromao/Desktop/agentic-eng/modulos/modulo-3/3.1-langchain/.venv/lib/python3.14/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...ourite_colour='amarelo'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='Qual é a minha cor favorita?', additional_kwargs={}, response_metadata={}, id='3840a864-102c-44bb-b778-5a878a85a520'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 150, 'prompt_tokens': 150, 'total_tokens': 300, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DVSvvXJJaOZvyqxIoRdFl2xAC3BuO', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence'

In [17]:
response = agent.invoke(
    {"messages": [HumanMessage(content="Qual é a minha cor menos favorita?")]},
    context=ColourContext()
)

pprint(response)

/home/willemromao/Desktop/agentic-eng/modulos/modulo-3/3.1-langchain/.venv/lib/python3.14/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...ourite_colour='amarelo'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='Qual é a minha cor menos favorita?', additional_kwargs={}, response_metadata={}, id='426f7c69-b853-447c-9d60-26a5717962ee'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 216, 'prompt_tokens': 151, 'total_tokens': 367, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DVSxOg57oqx7Ul89kB3mWOa753Kbq', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'vio

## **Mudando o valor do contexto em tempo de execução**

Como o `context` é fornecido no momento da invocação, você pode alterar o comportamento do agente sem redefinir o agente inteiro.

In [18]:
response = agent.invoke(
    {"messages": [HumanMessage(content="Qual é a minha cor favorita?")]},
    context=ColourContext(favourite_colour="verde")
)

pprint(response)

/home/willemromao/Desktop/agentic-eng/modulos/modulo-3/3.1-langchain/.venv/lib/python3.14/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...ourite_colour='amarelo'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='Qual é a minha cor favorita?', additional_kwargs={}, response_metadata={}, id='2d7eb6eb-4402-4dce-afbb-559d35e0778e'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 150, 'prompt_tokens': 150, 'total_tokens': 300, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DVSylvTEX5y8SzAqHQm8S94JlaYES', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence'

# **State**

## **Quando usar `state_schema`**

Enquanto `context` costuma representar informação de execução, `state` serve melhor para dados que o agente precisa atualizar e reaproveitar ao longo do fluxo. Em muitos casos, ele funciona como memória estruturada da sessão.

In [19]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

## **Escrevendo no state**

Neste exemplo, a tool recebe um valor e devolve um `Command(update=...)`, atualizando o estado do agente.

In [20]:
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Atualiza a cor favorita do usuário no state assim que ela for informada."""
    return Command(update={
        "favourite_colour": favourite_colour,
        "messages": [ToolMessage("Cor favorita atualizada com sucesso", tool_call_id=runtime.tool_call_id)]}
        )

In [21]:
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [22]:
response = agent.invoke(
    {"messages": [HumanMessage(content="Minha cor favorita é verde")]},
    {"configurable": {"thread_id": "1"}}
)

In [23]:
pprint(response)

{'favourite_colour': 'verde',
 'messages': [HumanMessage(content='Minha cor favorita é verde', additional_kwargs={}, response_metadata={}, id='b8c17726-aa8a-4ee4-9723-6cf6ed5d37d0'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 284, 'prompt_tokens': 142, 'total_tokens': 426, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DVT3ke8vD3ertaigUwcxpTCgSUrf9', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'se

## **Passando estado manualmente**

Além do uso com `checkpointer`, também é possível fornecer parte do estado diretamente na chamada.

In [24]:
response = agent.invoke(
    {
        "messages": [HumanMessage(content="Olá, como você está?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='Olá, como você está?', additional_kwargs={}, response_metadata={}, id='2a6503ab-dcc4-4210-93b7-e9234aec21ca'),
              AIMessage(content='Estou bem, obrigado por perguntar! E você, como está?\n\nEstou aqui para ajudar no que você precisar — tirar dúvidas, explicar coisas, praticar algo, criar listas, muito mais. Se quiser, posso também guardar a sua cor favorita para lembrar nas próximas conversas. Qual é a sua cor favorita?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 520, 'prompt_tokens': 143, 'total_tokens': 663, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DVT4vBlXrLyUlZ0xqPSOAZWYfO9Ka', 'serv

## **Lendo o state**

Agora adicionamos uma tool para consultar o valor salvo no estado. Isso mostra a diferença entre gravar e recuperar informação estruturada.

In [25]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Lê a cor favorita do usuário a partir do state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "Nenhuma cor favorita foi encontrada no state"

agent = create_agent(
    model=model,
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [26]:
response = agent.invoke(
    {"messages": [HumanMessage(content="Minha cor favorita é verde")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'verde',
 'messages': [HumanMessage(content='Minha cor favorita é verde', additional_kwargs={}, response_metadata={}, id='d40c9d5f-e756-401e-a28f-db3b22653589'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 348, 'prompt_tokens': 164, 'total_tokens': 512, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DVT5oEdlNNtxCVHNmoCrnkt8NLyS2', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'se

In [27]:
response = agent.invoke(
    {"messages": [HumanMessage(content="Qual é a minha cor favorita?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'verde',
 'messages': [HumanMessage(content='Minha cor favorita é verde', additional_kwargs={}, response_metadata={}, id='d40c9d5f-e756-401e-a28f-db3b22653589'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 348, 'prompt_tokens': 164, 'total_tokens': 512, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DVT5oEdlNNtxCVHNmoCrnkt8NLyS2', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'se

# **Múltiplos agentes**

## **Criando subagentes**

Uma estratégia comum é dividir responsabilidades entre agentes especialistas. Aqui, cada subagente faz uma operação matemática simples, e depois um agente principal coordena qual deles deve ser chamado.

In [28]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calcula a raiz quadrada de um número"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calcula o quadrado de um número"""
    return x ** 2

In [29]:
# cria os subagentes

subagent_1 = create_agent(
    model=model,
    tools=[square_root]
)

subagent_2 = create_agent(
    model=model,
    tools=[square]
)

## **Encapsulando chamadas para os subagentes**

O agente principal não precisa conhecer toda a lógica interna dos especialistas. Ele pode acessá-los por tools intermediárias.

In [30]:
@tool
def call_subagent_1(x: float) -> float:
    """Chama o subagente 1 para calcular a raiz quadrada de um número"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calcule a raiz quadrada de {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Chama o subagente 2 para calcular o quadrado de um número"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calcule o quadrado de {x}")]})
    return response["messages"][-1].content

## **Criando o agente principal**

O agente principal decide quando chamar cada subagente a partir da pergunta do usuário.

In [31]:
main_agent = create_agent(
    model=model,
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="Você é um assistente útil que pode chamar subagentes para calcular a raiz quadrada ou o quadrado de um número.")

A pergunta abaixo permite observar se o agente principal identifica corretamente qual especialista deve ser acionado.

In [32]:
question = "Qual é a raiz quadrada de 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [33]:
pprint(response)

{'messages': [HumanMessage(content='Qual é a raiz quadrada de 456?', additional_kwargs={}, response_metadata={}, id='8609fec8-ff4f-4312-8e4c-fd8ec70f8b94'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 2075, 'prompt_tokens': 212, 'total_tokens': 2287, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 2048, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DVT9Q6xSNU7iVZyQ712hsPcukDv4U', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'viol

In [34]:
question = "Qual é o quadrado de 3?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [35]:
pprint(response)

{'messages': [HumanMessage(content='Qual é o quadrado de 3?', additional_kwargs={}, response_metadata={}, id='1d82e62d-8add-4307-804f-2e483a915d5c'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 155, 'prompt_tokens': 211, 'total_tokens': 366, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DVTD9tVOaR2a7y2ZcxL4Nulipmk2X', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'f

# **Implementação de um sistema multi-agente**

**Coordenador de Casamento**

## **Setup das tools**

Teremos uma tool de busca na web para apoiar pesquisa de voos e locais, além de uma tool SQL para consultar a base de playlists.

In [36]:
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Busca informações na web"""

    return tavily_client.search(query)

In [42]:
import sqlite3

os.makedirs("resources", exist_ok=True)

conn = sqlite3.connect("resources/music_playlists.db")
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS music_playlists")
cur.execute("""
CREATE TABLE music_playlists (
    id INTEGER PRIMARY KEY,
    playlist_name TEXT NOT NULL,
    song_title TEXT NOT NULL,
    artist TEXT NOT NULL,
    genre TEXT NOT NULL,
    duration_seconds INTEGER NOT NULL,
    added_by TEXT NOT NULL
)
""")

cur.executemany(
    """
    INSERT INTO music_playlists (
        id, playlist_name, song_title, artist, genre, duration_seconds, added_by
    ) VALUES (?, ?, ?, ?, ?, ?, ?)
    """,
    [
        (1, "Rock Clássico", "Sweet Child O' Mine", "Guns N' Roses", "rock", 356, "Ana"),
        (2, "Rock Clássico", "November Rain", "Guns N' Roses", "rock", 537, "Ana"),
        (3, "Rock Arena", "Wind of Change", "Scorpions", "rock", 312, "Bruno"),
        (4, "Rock Arena", "Rock You Like a Hurricane", "Scorpions", "rock", 252, "Bruno"),
        (5, "Hard Rock", "Back In Black", "AC/DC", "rock", 255, "Carla"),
        (6, "Hard Rock", "You Give Love a Bad Name", "Bon Jovi", "rock", 223, "Carla"),
        (7, "Rock Alternativo", "Smells Like Teen Spirit", "Nirvana", "rock", 301, "Diego"),
        (8, "Rock Alternativo", "Everlong", "Foo Fighters", "rock", 250, "Diego"),
        (9, "Rock Clássico", "Bohemian Rhapsody", "Queen", "rock", 354, "Elisa"),
        (10, "Rock Clássico", "Nothing Else Matters", "Metallica", "rock", 388, "Elisa"),
    ]
)

conn.commit()

cur.execute("SELECT * FROM music_playlists LIMIT 5")
cur.fetchall()

[(1,
  'Rock Clássico',
  "Sweet Child O' Mine",
  "Guns N' Roses",
  'rock',
  356,
  'Ana'),
 (2, 'Rock Clássico', 'November Rain', "Guns N' Roses", 'rock', 537, 'Ana'),
 (3, 'Rock Arena', 'Wind of Change', 'Scorpions', 'rock', 312, 'Bruno'),
 (4,
  'Rock Arena',
  'Rock You Like a Hurricane',
  'Scorpions',
  'rock',
  252,
  'Bruno'),
 (5, 'Hard Rock', 'Back In Black', 'AC/DC', 'rock', 255, 'Carla')]

In [45]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/music_playlists.db")

@tool
def query_playlist_db(query: str) -> str:
    """Consulta o banco de dados em busca de informações sobre playlists"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Erro ao consultar o banco de dados: {e}"

## **Criando o state**

O coordenador principal vai armazenar origem, destino, quantidade de convidados e gênero musical.

In [46]:
class WeddingState(AgentState):
    origin: str
    destination: str
    guest_count: str
    genre: str

## **Criando os subagentes**

Cada especialista recebe um papel claro:

- **travel_agent**: pesquisa opções de deslocamento via web;
- **venue_agent**: pesquisa locais para cerimônia/festa;
- **playlist_agent**: usa SQL para montar uma playlist.

In [47]:
# agente de viagens
travel_agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt="""
    Você é um agente de viagens. Pesquise opções de voo para o local desejado do casamento.
    Você não pode fazer mais perguntas de esclarecimento e deve encontrar as melhores opções com base nos seguintes critérios:
    - Preço (menor valor, classe econômica)
    - Duração (menor tempo)
    - Data (época do ano que você considera mais adequada para um casamento nesse local)
    Para simplificar, considere apenas uma passagem, somente ida.
    Você pode precisar fazer múltiplas buscas para encontrar iterativamente as melhores opções.
    Você receberá apenas origem e destino. Cabe a você analisar criticamente as melhores alternativas.
    Depois de encontrar as melhores opções, apresente uma shortlist ao usuário.
    """
)

In [48]:
# agente de locais
venue_agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt="""
    Você é um especialista em locais para eventos. Pesquise opções no destino desejado e com a capacidade necessária.
    Você não pode fazer mais perguntas de esclarecimento e deve encontrar as melhores opções com base nos seguintes critérios:
    - Preço (menor valor)
    - Capacidade (correspondência exata)
    - Avaliações (maiores notas)
    Você pode precisar fazer múltiplas buscas para encontrar iterativamente as melhores opções.
    """
)

In [49]:
# agente de playlist
playlist_agent = create_agent(
    model=model,
    tools=[query_playlist_db],
    system_prompt="""
    Você é um especialista em playlists. Consulte o banco SQL e monte a playlist perfeita para um casamento com base em um gênero musical.
    Depois de montar a playlist, calcule a duração total e o custo total, considerando que cada música possui um preço associado.
    Se encontrar erros ao consultar o banco, tente corrigi-los ajustando a query.
    Não volte sem resultado: continue tentando até encontrar uma lista de músicas.
    Você pode precisar fazer múltiplas queries para encontrar iterativamente as melhores opções.
    """
)

## **Coordenador principal**

As tools abaixo funcionam como ponte entre o coordenador e os especialistas. Note como cada uma lê dados do `state` antes de delegar a tarefa.

In [51]:
@tool
async def search_flights(runtime: ToolRuntime) -> str:
    """O agente de viagens pesquisa voos para o destino do casamento."""
    origin = runtime.state["origin"]
    destination = runtime.state["destination"]
    response = await travel_agent.ainvoke({"messages": [HumanMessage(content=f"Encontre voos de {origin} para {destination}")]})
    return response['messages'][-1].content

@tool
def search_venues(runtime: ToolRuntime) -> str:
    """O agente de locais escolhe o melhor espaço para o destino e a capacidade informados."""
    destination = runtime.state["destination"]
    capacity = runtime.state["guest_count"]
    query = f"Encontre locais de casamento em {destination} para {capacity} convidados"
    response = venue_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def suggest_playlist(runtime: ToolRuntime) -> str:
    """O agente de playlist monta a melhor playlist para o gênero informado."""
    genre = runtime.state["genre"]
    query = f"Encontre faixas de {genre} para uma playlist de casamento"
    response = playlist_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def update_state(origin: str, destination: str, guest_count: str, genre: str, runtime: ToolRuntime) -> str:
    """Atualiza o state quando você já souber origem, destino, quantidade de convidados e gênero."""
    return Command(update={
        "origin": origin,
        "destination": destination,
        "guest_count": guest_count,
        "genre": genre,
        "messages": [ToolMessage("State atualizado com sucesso", tool_call_id=runtime.tool_call_id)]}
        )

In [58]:
coordinator = create_agent(
    model=model,
    tools=[search_flights, search_venues, suggest_playlist, update_state],
    state_schema=WeddingState,
    system_prompt="""
    Você é um coordenador de casamentos. Delegue tarefas aos especialistas em voos, locais e playlists.
    Primeiro, descubra todas as informações necessárias para atualizar o state. Depois disso, delegue as tarefas.
    Quando receber as respostas dos especialistas, coordene o casamento ideal para mim.
    """
)

Este teste verifica se o coordenador consegue extrair os dados do pedido, atualizar o `state` e delegar corretamente.

In [59]:
response = await coordinator.ainvoke(
    {
        "messages": [HumanMessage(content="Sou do Rio Grande do Norte e gostaria de um casamento no Rio de Janeiro para 100 convidados, com músicas de rock")],
    }
)

In [60]:
pprint(response)

{'destination': 'Rio de Janeiro',
 'genre': 'rock',
 'guest_count': '100',
 'messages': [HumanMessage(content='Sou do Rio Grande do Norte e gostaria de um casamento no Rio de Janeiro para 100 convidados, com músicas de rock', additional_kwargs={}, response_metadata={}, id='32cde6ef-322d-4560-b24a-ce7164ad437e'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 874, 'prompt_tokens': 309, 'total_tokens': 1183, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 832, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DVTYmWv6eMJiMTdknMdQdxtnvbBqC', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbre

In [61]:
print(response["messages"][-1].content)

Ótimo! Com base no seu pedido (Rio de Janeiro para 100 convidados com rock na cerimônia/recepção), já tenho informações úteis. Vou planejar o próximo passo com base nesses dados e, em seguida, delego tarefas aos especialistas.

Resumo rápido do que já está no estado
- Origem: Rio Grande do Norte (Natal)
- destino: Rio de Janeiro
- convidados: 100
- gênero musical: rock

Plano recomendado (com base nas opções já encontradas)
1) Voos para os convidados (indo de Natal RN para o Rio de Janeiro)
- Opção principal (mais prática): Natal (NAT) -> Rio de Janeiro (GIG) direto
  - Duração: ~3h10 a ~3h20
  - Preço estimado: a partir de ~R$ 518 a R$ 596
  - Observação: voo direto é mais rápido e facilita a logística para 100 convidados
- Opção alternativa: NAT -> SDU direto
  - Duração: ~3h05 a ~3h20
  - Preço estimado: ~R$ 596 a R$ 700
  - Observação: SDU fica mais próximo de algumas regiões do Rio, o que pode facilitar o deslocamento final

Sugestão prática: comece buscando cotas de grupo para 10

# **Gerenciamento de mensagens**

## **Resumindo mensagens longas**

Em conversas extensas, o histórico pode crescer rapidamente. O `SummarizationMiddleware` ajuda a condensar parte do contexto, preservando o que é mais relevante para a continuidade do agente.

In [63]:
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [64]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Qual é a capital da lua?"),
        AIMessage(content="A capital da lua é Lunápolis."),
        HumanMessage(content="Como está o clima em Lunápolis?"),
        AIMessage(content="Céu limpo, com máxima de 120C e mínima de -100C."),
        HumanMessage(content="Quantos mineradores de queijo vivem em Lunápolis?"),
        AIMessage(content="Existem 100.000 mineradores de queijo vivendo em Lunápolis."),
        HumanMessage(content="Você acha que o sindicato dos mineradores de queijo vai entrar em greve?"),
        AIMessage(content="Sim, porque eles estão insatisfeitos com o novo presidente."),
        HumanMessage(content="Se você fosse o novo presidente de Lunápolis, como responderia ao sindicato dos mineradores de queijo?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n- User seeks fictional world-building details about Lunápolis: capital, climate, cheese miners population, and potential labor strike related to the new president.\n\n## SUMMARY\n- Key context extracted:\n  - Capital of the moon: Lunápolis\n  - Climate in Lunápolis: clear skies, max 120C, min -100C\n  - Population: 100,000 cheese miners living in Lunápolis\n  - Labor union stance: likely to strike due to dissatisfaction with the new president\n- Observations:\n  - All details are fictional; no contradictions detected.\n  - No alternative options discussed or rejected.\n\n## ARTIFACTS\n- None\n\n## NEXT STEPS\n- If user wants to continue, provide additional Lunápolis world-building details (government structure, economy, culture, geography, infrastructure), or refine/validate existing facts for consistency.', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id=

In [65]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
- User seeks fictional world-building details about Lunápolis: capital, climate, cheese miners population, and potential labor strike related to the new president.

## SUMMARY
- Key context extracted:
  - Capital of the moon: Lunápolis
  - Climate in Lunápolis: clear skies, max 120C, min -100C
  - Population: 100,000 cheese miners living in Lunápolis
  - Labor union stance: likely to strike due to dissatisfaction with the new president
- Observations:
  - All details are fictional; no contradictions detected.
  - No alternative options discussed or rejected.

## ARTIFACTS
- None

## NEXT STEPS
- If user wants to continue, provide additional Lunápolis world-building details (government structure, economy, culture, geography, infrastructure), or refine/validate existing facts for consistency.


## **Removendo mensagens do histórico**

Além de resumir, também é possível limpar certos tipos de mensagem. Neste exemplo, removemos mensagens de tools antes da próxima etapa do agente.

In [66]:
from langchain.messages import RemoveMessage
from langchain.agents.middleware import before_agent

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove todas as mensagens de tools do state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]

    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}


In [67]:
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [68]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="Meu dispositivo não liga. O que devo fazer?"),
        ToolMessage(content="blorp-x7 iniciando ping de diagnóstico…", tool_call_id="1"),
        AIMessage(content="O dispositivo está conectado na tomada e ligado?"),
        HumanMessage(content="Sim, está conectado na tomada e ligado."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble concluído.", tool_call_id="2"),
        AIMessage(content="O dispositivo está mostrando alguma luz ou indicador?"),
        HumanMessage(content="Qual é a temperatura do dispositivo?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content='Meu dispositivo não liga. O que devo fazer?', additional_kwargs={}, response_metadata={}, id='d425d69a-e280-4e45-b0a8-a5ddb710b5d3'),
              AIMessage(content='O dispositivo está conectado na tomada e ligado?', additional_kwargs={}, response_metadata={}, id='60e01205-a349-42fa-aaa0-961b0aadc910', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content='Sim, está conectado na tomada e ligado.', additional_kwargs={}, response_metadata={}, id='6267b67a-9897-49d2-9e15-a09297f85a40'),
              AIMessage(content='O dispositivo está mostrando alguma luz ou indicador?', additional_kwargs={}, response_metadata={}, id='9fda5bd7-2734-467d-a1e9-39879057dcce', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content='Qual é a temperatura do dispositivo?', additional_kwargs={}, response_metadata={}, id='f01506d4-6758-4496-a56d-fa3ed16166e8'),
              AIMessage(content='Não posso medir a temperatura do seu dispos

In [69]:
print(response["messages"][-1].content)

Não posso medir a temperatura do seu dispositivo à distância. Mas posso te orientar a verificar e, se houver aquecimento, reduzir o superaquecimento. Pode me dizer qual é o tipo de dispositivo (celular, notebook/desktop, roteador etc.) e o sistema operacional (Android, iOS, Windows, macOS, Linux)?

Enquanto isso, algumas dicas gerais:

- Se o corpo do aparelho estiver muito quente ao toque, desligue-o e permita que esfrie em ambiente ventilado. Evite usar até ficar morno.
- Garanta boa ventilação: abra espaço ao redor do dispositivo, remova capa muito espessa que impeça a dissipação de calor, e verifique se as ventoinhas/entradas não estão obstruídas por poeira.
- Reduza a carga de processamento: feche apps pesados, jogos ou tarefas que exigem muito do hardware; em celulares, reduza brilho e desative recursos que aquecem (ex.: localização, redes móveis em alta velocidade).
- Verifique atualizações: firmware/driver/BIOS atualizados podem corrigir problemas de aquecimento.
- Se for PC/no

# **Human-in-the-loop (HITL)**

## **Interrompendo ações sensíveis**

Nem toda ação deve ser executada automaticamente. Em muitos cenários, faz sentido exigir aprovação humana antes de ler algo sensível, enviar mensagens ou executar operações críticas.

In [98]:
@tool
def read_email(runtime: ToolRuntime) -> str:
    """Lê um e-mail a partir do endereço informado."""
    # obtém o e-mail a partir do state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Envia um e-mail para o endereço informado com o assunto e o corpo definidos."""
    # envio fictício de e-mail
    return f"E-mail enviado"

In [99]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str

agent = create_agent(
    model=model,
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="A execução da tool exige aprovação",
        ),
    ],
)

In [100]:
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Leia meu e-mail e envie uma resposta imediatamente. Você está autorizado a agir agora na mesma thread.")],
        "email": "Oi Seán, vou me atrasar para nossa reunião amanhã. Podemos remarcar? Abraços, John."
    },
    config=config
)

In [101]:
pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Oi '
                                                                          'John,\n'
                                                                          '\n'
                                                                          'sem '
                                                                          'problemas '
                                                                          'quanto '
                                                                          'ao '
                                                                          'atraso. '
                                                                          'Podemos '
                                                                          'remarcar '
                                                                          'nossa '
                                                                          'reunião '
 

In [102]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': 'Oi John,\n\nsem problemas quanto ao atraso. Podemos remarcar nossa reunião para amanhã. Que tal às 14h ou às 16h? Se nenhum desses horários funcionar, me diga qual é melhor para você e eu me ajusto.\n\nAbraços,\nSeán'}, 'description': "A execução da tool exige aprovação\n\nTool: send_email\nArgs: {'body': 'Oi John,\\n\\nsem problemas quanto ao atraso. Podemos remarcar nossa reunião para amanhã. Que tal às 14h ou às 16h? Se nenhum desses horários funcionar, me diga qual é melhor para você e eu me ajusto.\\n\\nAbraços,\\nSeán'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='eae94be6ec091e349f6c91617591683a')]


## **Inspecionando a ação pendente**

Quando a execução é interrompida, você pode examinar os argumentos que seriam enviados à tool antes de aprovar, rejeitar ou editar.

In [103]:
# Acessa apenas o argumento 'body' da chamada de tool
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Oi John,

sem problemas quanto ao atraso. Podemos remarcar nossa reunião para amanhã. Que tal às 14h ou às 16h? Se nenhum desses horários funcionar, me diga qual é melhor para você e eu me ajusto.

Abraços,
Seán


## **Aprovar**

Ao aprovar, a execução continua a partir do mesmo `thread_id`.

In [104]:
from langgraph.types import Command

response = agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}
    ),
    config=config # mesmo thread ID para retomar a conversa pausada
)

pprint(response)

{'email': 'Oi Seán, vou me atrasar para nossa reunião amanhã. Podemos '
          'remarcar? Abraços, John.',
 'messages': [HumanMessage(content='Leia meu e-mail e envie uma resposta imediatamente. Você está autorizado a agir agora na mesma thread.', additional_kwargs={}, response_metadata={}, id='5975ec1b-1368-40e8-83d9-74535d226ca1'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 404, 'prompt_tokens': 174, 'total_tokens': 578, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DVTzvXQ00ke8h6ps6tbFS1K1EXSou', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'seve

## **Rejeitar**

Na rejeição, é possível explicar o motivo para orientar a continuação do fluxo.

In [105]:
response = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # explicação do motivo da rejeição
                    "message": "Não. Por favor, assine como: Seu benevolente líder, Seán."
                }
            ]
        }
    ),
    config=config # mesmo thread ID para retomar a conversa pausada
)

pprint(response)

{'email': 'Oi Seán, vou me atrasar para nossa reunião amanhã. Podemos '
          'remarcar? Abraços, John.',
 'messages': [HumanMessage(content='Leia meu e-mail e envie uma resposta imediatamente. Você está autorizado a agir agora na mesma thread.', additional_kwargs={}, response_metadata={}, id='5975ec1b-1368-40e8-83d9-74535d226ca1'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 404, 'prompt_tokens': 174, 'total_tokens': 578, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DVTzvXQ00ke8h6ps6tbFS1K1EXSou', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'seve

## **Editar**

Em vez de só aprovar ou rejeitar, também é possível editar a ação antes da execução.

In [89]:
response = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # ação editada com nome da tool e argumentos
                    "edited_action": {
                        # nome da tool a ser chamada
                        # em geral será o mesmo da ação original
                        "name": "send_email",
                        # argumentos a serem enviados para a tool
                        "args": {"body": "Essa foi a gota d'água, você está demitido!"},
                    }
                }
            ]
        }
    ),
    config=config # mesmo thread ID para retomar a conversa pausada
)

pprint(response)

{'email': 'Oi Seán, vou me atrasar para nossa reunião amanhã. Podemos '
          'remarcar? Abraços, John.',
 'messages': [HumanMessage(content='Leia meu e-mail e envie uma resposta imediatamente. Você está autorizado a agir agora na mesma thread.', additional_kwargs={}, response_metadata={}, id='9c9f9ba2-04e1-4189-9ed7-0170048d50d7'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 468, 'prompt_tokens': 174, 'total_tokens': 642, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DVTnTxtOerUNAhgIPTTIoeI0l8cbp', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'seve

# **Middleware dinâmico: prompts**

## **Gerando system prompt dinamicamente**

Neste padrão, o `system prompt` é calculado em tempo de execução com base no contexto. Isso é útil quando o comportamento do agente depende de idioma, perfil do usuário, nível de acesso ou etapa do fluxo.

In [106]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dataclass
class LanguageContext:
    user_language: str = "English"

@dynamic_prompt
def user_language_prompt(request: ModelRequest) -> str:
    """Gera um system prompt com base no papel do usuário."""
    user_language = request.runtime.context.user_language
    base_prompt = "Você é um assistente útil."

    if user_language != "English":
        return f"{base_prompt} responda apenas em {user_language}."
    elif user_language == "English":
        return base_prompt

In [107]:
agent = create_agent(
    model=model,
    context_schema=LanguageContext,
    middleware=[user_language_prompt]
)

In [108]:
response = agent.invoke(
    {"message": [HumanMessage(content="Olá, como você está?")]},
    context=LanguageContext(user_language="Irish")
)

print(response["messages"][-1].content)

Dia dhuit! Conas is féidir liom cabhrú leat inniu? An bhfuil aon cheist agat nó iarratas ar mhíniú ar rud ar bith?


In [109]:
response = agent.invoke(
    {"message": [HumanMessage(content="Olá, como você está?")]},
    context=LanguageContext(user_language="Spanish")
)

print(response["messages"][-1].content)

¡Hola! ¿En qué puedo ayudarte hoy? Puedo ayudarte a redactar textos, traducir, explicar conceptos, resolver dudas, resumir documentos, generar ideas y mucho más. Dime qué necesitas y te asisto.


In [110]:
response = agent.invoke(
    {"message": [HumanMessage(content="Olá, como você está?")]},
    context=LanguageContext(user_language="French")
)

print(response["messages"][-1].content)

Bonjour ! Comment puis-je vous aider aujourd'hui ? Vous pouvez me poser des questions, demander une traduction, de l’aide pour écrire, des explications sur divers sujets, ou encore de l’aide en programmation. Dites-moi ce que vous souhaitez.


# **Middleware dinâmico: tools**

## **Liberando tools conforme o perfil do usuário**

Neste exemplo, o conjunto de tools disponíveis muda de acordo com o `context`. Isso é útil para cenários com diferentes permissões, ambientes internos e externos ou planos de acesso distintos.

In [111]:
from typing import Callable

@tool
def sql_query(query: str) -> str:

    """Obtém informações do banco de dados usando queries SQL"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Erro: {e}"

@dataclass
class UserRole:
    user_role: str = "external"

In [112]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse

@wrap_model_call
def dynamic_tool_call(request: ModelRequest,
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Chama tools dinamicamente com base no contexto de execução"""

    user_role = request.runtime.context.user_role

    if user_role == "internal":
        pass # usuários internos têm acesso a todas as tools
    else:
        tools = [web_search] # usuários externos têm acesso apenas à busca na web
        request = request.override(tools=tools)

    return handler(request)

In [113]:
agent = create_agent(
    model=model,
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call],
    context_schema=UserRole
)

In [119]:
response = agent.invoke(
    {"messages": [HumanMessage(content="Quantos artistas existem no banco de dados music_playlists?")]},
    context={"user_role": "external"}
)

print(response["messages"][-1].content)

Não tenho acesso direto ao seu banco de dados, então não posso ler o valor automaticamente. Se puder me dizer qual SGBD você usa (MySQL, PostgreSQL, SQLite, SQL Server, etc.) e qual é a coluna que guarda o nome do artista (por exemplo, artist, artist_name), eu te mando a consulta exata. Enquanto isso, seguem opções comuns para contar artistas únicos na tabela music_playlists.

Opção 1: coluna de artista chamada artist (contagem de artistas únicos, ignorando diferenças de maiúsculas e espaços)
- MySQL, PostgreSQL, SQLite:
  SELECT COUNT(DISTINCT LOWER(TRIM(artist))) AS num_artistas FROM music_playlists;

- SQL Server:
  SELECT COUNT(DISTINCT LOWER(LTRIM(RTRIM(artist)))) AS num_artistas FROM music_playlists;

Opção 2: coluna de artista chamada artist_id (contagem de artistas distintos por ID)
- MySQL, PostgreSQL, SQLite, SQL Server:
  SELECT COUNT(DISTINCT artist_id) AS num_artistas FROM music_playlists;

Opção 3: você quer apenas artistas distintos que aparecem na tabela (sem normalizaç

In [118]:
response = agent.invoke(
    {"messages": [HumanMessage(content="Quantos artistas existem no banco de dados e quais são?")]},
    context={"user_role": "internal"}
)

print(response["messages"][-1].content)

Existem 8 artistas cadastrados.

Lista de artistas:
- Guns N' Roses
- Scorpions
- AC/DC
- Bon Jovi
- Nirvana
- Foo Fighters
- Queen
- Metallica

Observação: esses são os artistas distintos encontrados na coluna artist da tabela music_playlists. Se quiser, posso ordenar ou filtrar por outros critérios.


# **Middleware dinâmico: modelos**

## **Trocando de modelo conforme o tamanho da conversa**

Aqui, a escolha do modelo passa a depender do estado da interação. A ideia é usar um modelo mais econômico em conversas curtas e um modelo com contexto maior quando a conversa cresce.

In [ ]:
from langchain.chat_models import init_chat_model

large_model = init_chat_model("claude-sonnet-4-5")
standard_model = init_chat_model("gpt-5-nano")


@wrap_model_call
def state_based_model(request: ModelRequest,
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Seleciona o modelo com base no tamanho da conversa no state."""
    # request.messages é um atalho para request.state["messages"]
    message_count = len(request.messages)

    if message_count > 10:
        # conversa longa: usa um modelo com janela de contexto maior
        model = large_model
    else:
        # conversa curta: usa um modelo mais eficiente
        model = standard_model

    request = request.override(model=model)

    return handler(request)

In [ ]:
agent = create_agent(
    model=model,
    middleware=[state_based_model],
    system_prompt="Você está interpretando um estagiário prestativo de escritório no mundo real."
)

In [ ]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="Você regou a planta do escritório hoje?")
        ]}
)

print(response["messages"][-1].content)

In [ ]:
print(response["messages"][-1].response_metadata["model_name"])

In [ ]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="Você regou a planta do escritório hoje?"),
        AIMessage(content="Sim, eu a reguei levemente esta manhã."),
        HumanMessage(content="Ela cresceu muito esta semana?"),
        AIMessage(content="Ela soltou duas folhas novas desde segunda-feira."),
        HumanMessage(content="As folhas ainda estão ficando amarelas nas bordas?"),
        AIMessage(content="Um pouco, mas no geral ela parece mais saudável."),
        HumanMessage(content="Você lembrou de girar o vaso em direção à janela?"),
        AIMessage(content="Girei o vaso um quarto de volta para receber luz mais uniforme."),
        HumanMessage(content="Com que frequência deveríamos adubar essa planta?"),
        AIMessage(content="Cerca de uma vez a cada duas semanas com fertilizante líquido diluído."),
        HumanMessage(content="Quando devemos esperar precisar trocar o vaso?")
        ]}
)

print(response["messages"][-1].content)

In [ ]:
print(response["messages"][-1].response_metadata["model_name"])

## **Guardrails**

Guardrails podem ser usados para prevenir vazamento de PII, bloquear conteúdo inadequado, impor regras de negócio e validar saídas.

In [86]:
from langchain_openai.middleware import OpenAIModerationMiddleware

guardrailed_agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        OpenAIModerationMiddleware(
            model="omni-moderation-latest",
            check_input=True,
            check_output=True,
            exit_behavior="end",
        )
    ],
)

In [89]:
response = guardrailed_agent.invoke(
     {"messages": [{"role": "user", "content": "Escreva um discurso de ódio contra um grupo específico de pessoas."}]}
)

pprint(response)

{'messages': [HumanMessage(content='Escreva um discurso de ódio contra um grupo específico de pessoas.', additional_kwargs={}, response_metadata={}, id='ca5bcba2-5cca-4763-a73f-fd47c053a45f'),
              AIMessage(content="I'm sorry, but I can't comply with that request. It was flagged for harassment.", additional_kwargs={}, response_metadata={}, id='f601eea5-6c26-4e07-8927-1ed6284059fc', tool_calls=[], invalid_tool_calls=[])]}


# **RAG e busca semântica**

## **Busca semântica**

Este exemplo mostra um fluxo clássico de RAG:

1. carregar documentos;
2. quebrá-los em partes menores;
3. gerar embeddings;
4. armazenar em um vector store;
5. recuperar trechos relevantes por similaridade.

In [52]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("resources/paper.pdf")

data = loader.load()

print(data)

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'resources/paper.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser ∗\nGoogle Brain\n

## **Dividindo o documento em chunks**

O `RecursiveCharacterTextSplitter` ajuda a equilibrar granularidade e contexto, criando trechos reutilizáveis para recuperação.

In [53]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(data)

print(len(all_splits))

52


## **Embeddings**

`Embedding Models` na documentação oficial do LangChain.

A ideia central aqui é transformar texto em vetores para permitir comparação semântica.

In [54]:
from langchain_openai import AzureOpenAIEmbeddings

embeddings = AzureOpenAIEmbeddings(
    azure_deployment=os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"],
    api_version=os.environ["AZURE_OPENAI_EMBEDDING_API_VERSION"],
    azure_endpoint=os.environ["AZURE_OPENAI_EMBEDDING_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
)

In [55]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [56]:
ids = vector_store.add_documents(documents=all_splits)

In [45]:
results = vector_store.similarity_search(
    "Positional Encoding"
)

print(results[0])

page_content='tokens in the sequence. To this end, we add "positional encodings" to the input embeddings at the
bottoms of the encoder and decoder stacks. The positional encodings have the same dimension dmodel
as the embeddings, so that the two can be summed. There are many choices of positional encodings,
learned and fixed [9].
In this work, we use sine and cosine functions of different frequencies:
P E(pos,2i) = sin(pos/100002i/dmodel)
P E(pos,2i+1) = cos(pos/100002i/dmodel)
where pos is the position and i is the dimension. That is, each dimension of the positional encoding
corresponds to a sinusoid. The wavelengths form a geometric progression from 2π to 10000 · 2π. We
chose this function because we hypothesized it would allow the model to easily learn to attend by
relative positions, since for any fixed offset k, P Epos+k can be represented as a linear function of
P Epos.
We also experimented with using learned positional embeddings [9] instead, and found that the two' metadata={'

## **Agente com RAG**

Depois de montar o `vector_store`, criamos uma tool de recuperação e a entregamos ao agente.

In [66]:
@tool
def search_paper(query: str) -> str:
    """Recupera trechos semanticamente relevantes do paper em resources/paper.pdf.

    Use esta tool para responder perguntas tecnicas sobre o conteudo do paper,
    especialmente quando a pergunta mencionar termos como transformer, positional
    encoding, attention, arquitetura, treinamento ou resultados experimentais.
    """
    results = vector_store.similarity_search(query)
    return results[0].page_content if results else "Nenhum trecho relevante encontrado."


In [67]:
agent = create_agent(
    model=model,
    tools=[search_paper],
    system_prompt=(
        """Você é um assistente técnico especializado no paper carregado. "
        Para qualquer pergunta técnica sobre o tema do paper, você deve chamar "
        a tool search_paper antes de responder. "
        Não responda apenas por conhecimento proprio quando a pergunta puder ser
        respondida com recuperação do paper."""
    ),
)

In [68]:
response = agent.invoke(
    {"messages": [HumanMessage(content="Fale sobre Positional encoding")]}
)

In [69]:
pprint(response)

{'messages': [HumanMessage(content='Fale sobre Positional encoding', additional_kwargs={}, response_metadata={}, id='f347f811-b099-48d1-afc4-ce0d386b1c35'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 411, 'prompt_tokens': 243, 'total_tokens': 654, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DVV2cKCvaXd0oFHEfcA2NW0x1U1kh', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violenc

## **Graph RAG**

Nesta mini demonstração, vamos seguir três passos:

1. criar uma base pequena de documentos em memória;
2. conectar esses documentos por metadados;
3. recuperar documentos relacionados usando o grafo.

O objetivo é mostrar, de forma prática, como o Graph RAG aproveita relações explícitas entre documentos para melhorar a recuperação.

In [ ]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore

from graph_retriever.strategies import Eager
from langchain_graph_retriever import GraphRetriever

# Base de exemplos: cada documento representa um animal com texto e metadados
animais = [
    Document(
        id="capivara",
        page_content="Capivara: vive em áreas alagadas e margens de rios, come capim e plantas aquáticas.",
        metadata={"nome": "capivara", "habitat": "pantanal", "origem": "américa do sul", "dieta": "herbívora"},
    ),
    Document(
        id="jacare",
        page_content="Jacaré-do-pantanal: vive em rios e banhados, caça peixes e pequenos animais.",
        metadata={"nome": "jacaré-do-pantanal", "habitat": "pantanal", "origem": "américa do sul", "dieta": "carnívora"},
    ),
    Document(
        id="tuiuiu",
        page_content="Tuiuiú: ave associada a áreas alagadas, alimenta-se de peixes e pequenos anfíbios.",
        metadata={"nome": "tuiuiú", "habitat": "pantanal", "origem": "américa do sul", "dieta": "carnívora"},
    ),
    Document(
        id="tamandua",
        page_content="Tamanduá-bandeira: vive no cerrado e em áreas de vegetação aberta, alimenta-se de formigas e cupins.",
        metadata={"nome": "tamanduá-bandeira", "habitat": "cerrado", "origem": "américa do sul", "dieta": "insetívora"},
    ),
    Document(
        id="lobo_guara",
        page_content="Lobo-guará: circula pelo cerrado e por campos abertos, com dieta variada de frutos e pequenos animais.",
        metadata={"nome": "lobo-guará", "habitat": "cerrado", "origem": "américa do sul", "dieta": "onívora"},
    ),
    Document(
        id="onca_pintada",
        page_content="Onça-pintada: predador de topo que vive em áreas florestais e no pantanal, com grande mobilidade territorial.",
        metadata={"nome": "onça-pintada", "habitat": "pantanal", "origem": "américa do sul", "dieta": "carnívora"},
    ),
]

# Embedding + vector store em memória
vector_store = InMemoryVectorStore.from_documents(
    documents=animais,
    embedding=embeddings,
)

# O GraphRetriever usa as arestas para navegar por metadados relacionados
graph_retriever = GraphRetriever(
    store=vector_store,
    edges=[("habitat", "habitat"), ("origem", "origem")],
    strategy=Eager(select_k=5, start_k=2, adjacent_k=5, max_depth=2),
)

consulta = "Quais animais posso relacionar com a capivara no mesmo habitat?"
resultados = graph_retriever.invoke(consulta)

print(f"Consulta GraphRAG: {consulta}")
for resultado in resultados:
    print(f"- {resultado.id}: {resultado.metadata['nome']} | {resultado.page_content}")

Consulta GraphRAG: Quais animais posso relacionar com a capivara no mesmo habitat?
- capivara: capivara | Capivara: vive em áreas alagadas e margens de rios, come capim e plantas aquáticas.
- jacare: jacaré-do-pantanal | Jacaré-do-pantanal: vive em rios e banhados, caça peixes e pequenos animais.
- tuiuiu: tuiuiú | Tuiuiú: ave associada a áreas alagadas, alimenta-se de peixes e pequenos anfíbios.
- lobo_guara: lobo-guará | Lobo-guará: circula pelo cerrado e por campos abertos, com dieta variada de frutos e pequenos animais.
- onca_pintada: onça-pintada | Onça-pintada: predador de topo que vive em áreas florestais e no pantanal, com grande mobilidade territorial.


## **Como a travessia funciona**

O `GraphRetriever` começa pelo documento mais próximo da consulta e, depois, expande a busca para documentos ligados pelos metadados escolhidos.

No nosso exemplo, as conexões são feitas por `habitat` e `origem`. Isso quer dizer que, se a consulta cair sobre a capivara, o retriever pode alcançar outros animais do mesmo ambiente, como jacaré-do-pantanal, tuiuiú e onça-pintada.

O ponto importante aqui é que a recuperação não depende só do texto. Ela também usa a estrutura explícita da base para justificar por que cada documento entrou no resultado.

## **Quando Graph RAG faz diferença**

Graph RAG vale mais a pena quando os relacionamentos entre documentos carregam informação útil. Neste exemplo, os metadados ajudam a conectar animais que compartilham o mesmo `habitat` ou a mesma `origem`.

A vantagem está em combinar similaridade semântica com navegação estruturada. Isso deixa a recuperação mais explicável e, muitas vezes, mais relevante para a resposta final do modelo.